## 1. Create Widget

In [0]:
import json
from datetime import datetime

default_metadata = {
    "table_id": 1,
    "table_name": "customers",
    "source_system": "sqlserver",
    "source_schema": "banking",
    "source_table": "customers",
    "source_path": "",
    "bronze_schema": "bronze",
    "silver_schema": "silver",
    "active_flag": True,
    "load_order": 1,
    "created_at": datetime.now().isoformat()
}

dbutils.widgets.text("table_metadata", json.dumps(default_metadata))

table_metadata = json.loads(dbutils.widgets.get("table_metadata"))

table_id = int(table_metadata["table_id"])

## 2. Read Table Parameters

In [0]:
params_df = (
    spark.table("banking.metadata.table_parameters")
         .filter(f"table_id = {table_id}")
)

## 3. Convert to Single JSON Object

In [0]:
rows = params_df.select("parameter_name", "parameter_value").collect()

parameters_dict = {
    row.parameter_name: row.parameter_value
    for row in rows
}

print("Parameters JSON Object:")
print(parameters_dict)

## 4. Set Databricks Task Value

In [0]:
dbutils.jobs.taskValues.set(
    key="table_parameters",
    value=parameters_dict
)

print("Task value 'table_parameters' has been set.")